In [ ]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, tf_load_sample_from_files
cnn_model=build_cnn_model(INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch01_valAcc0.5113.weights.h5')#('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset'
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', '*.npy'), recursive=True))
train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
train_dataset=train_dataset.take(100)
train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
  """A generator function that yields input data for quantization."""
  for input_value in train_dataset.batch(1).prefetch(tf.data.AUTOTUNE):
    # TFLite converter expects a list of input tensors
    yield [input_value[0]]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
converter.optimizations = [ tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8] 
# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)

I0000 00:00:1763048062.767248   95320 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1763048062.796288   95320 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1763048063.266376   95320 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1763048063.835563   95320 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel bi

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 312, 256, 32)   │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 312, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 312, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 312, 256, 64)   │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 312, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 312, 256, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 78, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 78, 64, 64)     │       102,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 78, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 78, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 39, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 384)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 89)             │        34,265 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 189,465 (740.10 KB)

 Trainable params: 189,145 (738.85 KB)

 Non-trainable params: 320 (1.25 KB)

INFO:tensorflow:Assets written to: /tmp/tmpx0tl7b35/assets


INFO:tensorflow:Assets written to: /tmp/tmpx0tl7b35/assets


Saved artifact at '/tmp/tmpx0tl7b35'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 312, 256, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 89), dtype=tf.float32, name=None)
Captures:
  140515085625936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085627088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085627664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085627280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085626320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085627472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085628048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085629008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085629776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140515085629968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1405150856286

/home/gerald/anaconda3/envs/gpu/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1763048065.308352   95320 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1763048065.308366   95320 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1763048065.308535   95320 reader.cc:83] Reading SavedModel from: /tmp/tmpx0tl7b35
I0000 00:00:1763048065.308912   95320 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1763048065.308916   95320 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpx0tl7b35
I0000 00:00:1763048065.312351   95320 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1763048065.313000   95320 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1763048065.338660   95320 loader.cc:220] Running initialization op on S